### Output Parsers
Output parsers in LangChain are specialized classes that transforms the output of Language models (LLMs) into a more suitable format.

In [2]:
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

True

In [7]:
model = GoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

#### CSV Parser

In [8]:
ouput_parser = CommaSeparatedListOutputParser()
format_instructions = ouput_parser.get_format_instructions()
prompt = PromptTemplate(
    template="List 5 places {places}.\n{format_instructions}",
    input_variables=["places"],
    partial_variables={"format_instructions": format_instructions}
)

In [9]:
chain = prompt | model | ouput_parser

In [10]:
chain.invoke({"places":"for summer tourism in India"})

['Manali', 'Shimla', 'Ladakh', 'Nainital', 'Munnar']

#### JSON Parser

In [13]:
from typing import List
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


In [14]:
class travel(BaseModel):
    place: str = Field(description="Name of the place")
    description: str = Field(description="Description of the place")
    activities: str = Field(
        description="List of activities to do at the place"
    )

In [15]:
travel_query = "Suggest a place in India for going on a trip this summer to avoid heat."

parser = JsonOutputParser(pydantic_object=travel)
prompt = PromptTemplate(
    template= "Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = prompt | model | parser
chain.invoke({"query": travel_query})

{'place': 'Munnar, Kerala',
 'description': "Munnar is a hill station in Kerala, South India. It's known for its sprawling tea plantations, lush green hills, and cool climate, making it a perfect escape from the summer heat.",
 'activities': 'Tea garden visits, trekking, wildlife sanctuary exploration, boating in Mattupetty Dam, and enjoying the scenic beauty.'}

In [16]:
#Without Pydantic
parser = JsonOutputParser()
prompt = PromptTemplate(
    template= "Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = prompt | model | parser
result = chain.invoke({"query": travel_query})
print(result)

{'suggestion': {'place': 'Munnar, Kerala', 'reason': 'Munnar, nestled in the Western Ghats, offers a refreshing escape from the summer heat with its lush green tea plantations, cool climate, and scenic beauty. The average temperature during summer ranges from 15°C to 25°C, making it significantly cooler than the plains. You can enjoy activities like trekking, exploring tea estates, visiting waterfalls, and enjoying the serene atmosphere.', 'activities': ['Trekking', 'Tea estate visits', 'Waterfall exploration (e.g., Attukal Waterfalls, Lakkam Waterfalls)', 'Boating in Mattupetty Dam', 'Visiting Eravikulam National Park (for Nilgiri Tahr sightings)', 'Exploring the Tata Tea Museum'], 'temperature_range': '15°C to 25°C (approximate)', 'travel_tips': ['Book accommodations in advance, especially during peak season.', 'Pack light woolens or jackets as the evenings can get chilly.', 'Carry comfortable walking shoes for trekking and exploring.', 'Stay hydrated by drinking plenty of water.', '

#### Structured Output Parser
It can be used when you want to return multiple fields. While the Pydantic/JSON parser is more powerful, this is useful for less powerful models

In [18]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser


In [21]:
response_schemas = [
    ResponseSchema(name="answer", description="answer to the user's query"),
    ResponseSchema(name="description", description="Detailed Description on the answer topic"),
    ResponseSchema(name="application", description="real world application of the answer topic")
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [23]:
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions}
)

In [24]:
chain = prompt | model | output_parser
chain.invoke({"query": "Name an invention in Healthcare that has caused a revolution in the field in 21st century."})

{'answer': 'CRISPR-Cas9 gene editing technology',
 'description': 'CRISPR-Cas9 (Clustered Regularly Interspaced Short Palindromic Repeats and CRISPR-associated protein 9) is a revolutionary gene editing technology that allows scientists to precisely alter DNA sequences within living organisms. It works by using a guide RNA molecule to direct the Cas9 enzyme to a specific location in the genome, where it cuts the DNA. This cut can then be used to disrupt a gene, insert a new gene, or correct a faulty gene. The precision and relative ease of use of CRISPR-Cas9 have made it a game-changer in various fields, particularly in healthcare.',
 'application': "CRISPR-Cas9 has numerous applications in healthcare, including:\n\n*   **Gene therapy:** Correcting genetic defects that cause diseases like cystic fibrosis, sickle cell anemia, and Huntington's disease.\n*   **Cancer treatment:** Developing new immunotherapies by engineering immune cells to target and destroy cancer cells. It can also be 